# 🧠 NeuroVerse — Motor Spiral & Wave Drawing Model Training
## Kaggle Notebook: Parkinson's Disease Detection via Drawing Analysis

### Clinical Background
**Spiral and wave drawing tests** are established clinical tools for detecting Parkinson's Disease (PD).
PD patients exhibit characteristic drawing impairments:
- **Tremor** — 4-6 Hz oscillations visible as wavy/jagged lines
- **Micrographia** — progressively smaller drawing amplitude
- **Bradykinesia** — slower drawing speed, uneven pressure
- **Rigidity** — loss of smooth curvature, angular deviations

### Datasets (Multi-Source — ~800+ images combined)

| # | Dataset | Source | Images | Types | Access |
|---|---------|--------|--------|-------|--------|
| 1 | **Parkinson's Drawings** (Zham et al., 2017) | Kaggle `kmader/parkinsons-drawings` | ~102 | Spiral + Wave | Direct Kaggle load |
| 2 | **Parkinson Drawing Spiral+Wave** | Kaggle `alihussainpk/parkinson-drawing-spiralwave` | ~200+ | Spiral + Wave | Direct Kaggle load |
| 3 | **Spiral Drawings PD** | Kaggle `vasanthakiranshahi/parkinson-spiral-drawings` | ~150+ | Spiral | Direct Kaggle load |
| 4 | **HandPD** (Pereira et al., 2016) | GitHub `biolab/handpd` | ~92 | Spiral + Meander | `git clone` (free) |
| 5 | **NewHandPD** (Pereira et al., 2016) | GitHub `biolab/handpd` | ~264 | Spiral + Meander | `git clone` (free) |

**Combined: ~800+ unique images → with 5x augmentation → ~4,000+ effective training samples**

### Architecture
- **EfficientNet-B0** (pretrained on ImageNet) with custom PD classification head
- **Dual-task**: Spiral classifier + Wave classifier → fused motor score
- **GradCAM** for XAI — highlights tremor regions in drawings
- **Data augmentation** (5x expansion with heavy transforms)

### Output
- `motor_spiral_model.pt` — PyTorch model for NeuroVerse backend
- UPDRS-aligned tremor severity scoring
- GradCAM visualization for doctor dashboard

## 1️⃣ Environment Setup & Multi-Dataset Loading

### Kaggle Datasets (add ALL of these in the sidebar → "Add Data")
1. `kmader/parkinsons-drawings` → `/kaggle/input/parkinsons-drawings/`
2. `alihussainpk/parkinson-drawing-spiralwave` → `/kaggle/input/parkinson-drawing-spiralwave/`
3. `vasanthakiranshahi/parkinson-spiral-drawings` → `/kaggle/input/parkinson-spiral-drawings/`

### Non-Kaggle Datasets (downloaded automatically via git clone)
4. **HandPD** + **NewHandPD** (Pereira et al., 2016) — from GitHub `biolab/handpd`

> **Note**: Add as many Kaggle datasets as available. The scanner handles missing datasets gracefully — if a dataset isn't added, it's skipped.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies & Import Libraries
# ============================================================

!pip install -q timm albumentations grad-cam seaborn scikit-learn tqdm

import os
import sys
import json
import random
import shutil
import warnings
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision import models

import timm  # EfficientNet
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# Reproducibility
# ═══════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"   PyTorch: {torch.__version__}")
print(f"   timm: {timm.__version__}")

# ═══════════════════════════════════════════════════════════
# Output directories
# ═══════════════════════════════════════════════════════════
OUTPUT_DIR = "/kaggle/working"
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# MULTI-DATASET REGISTRY
# Kaggle datasets: Add via sidebar → "Add Data"
# Non-Kaggle: Downloaded automatically below
# ═══════════════════════════════════════════════════════════
DATASET_REGISTRY = {
    # ── Kaggle datasets ──
    "kaggle_kmader": {
        "name": "Parkinson's Drawings (Zham et al., 2017)",
        "path": "/kaggle/input/parkinsons-drawings",
        "source": "kaggle",
        "slug": "kmader/parkinsons-drawings",
    },
    "kaggle_alihussain": {
        "name": "Parkinson Drawing Spiral+Wave",
        "path": "/kaggle/input/parkinson-drawing-spiralwave",
        "source": "kaggle",
        "slug": "alihussainpk/parkinson-drawing-spiralwave",
    },
    "kaggle_vasantha": {
        "name": "Spiral Drawings PD",
        "path": "/kaggle/input/parkinson-spiral-drawings",
        "source": "kaggle",
        "slug": "vasanthakiranshahi/parkinson-spiral-drawings",
    },
    # ── Non-Kaggle datasets (auto-downloaded) ──
    "handpd": {
        "name": "HandPD (Pereira et al., 2016)",
        "path": "/kaggle/working/handpd_data/HandPD",
        "source": "github",
        "url": "https://github.com/biolab/HandPD-dataset.git",
    },
    "newhandpd": {
        "name": "NewHandPD (Pereira et al., 2016)",
        "path": "/kaggle/working/handpd_data/NewHandPD",
        "source": "github",
        "url": "https://github.com/biolab/HandPD-dataset.git",  # same repo, different folder
    },
}

# ═══════════════════════════════════════════════════════════
# Download non-Kaggle datasets
# ═══════════════════════════════════════════════════════════
print("\n📥 Checking/downloading non-Kaggle datasets...")

HANDPD_CLONE_DIR = "/kaggle/working/handpd_data"
if not os.path.exists(HANDPD_CLONE_DIR):
    print("   Cloning HandPD dataset from GitHub...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/biolab/HandPD-dataset.git",
         HANDPD_CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("   ✅ HandPD repository cloned successfully")
    else:
        print(f"   ⚠️  HandPD clone failed: {result.stderr[:200]}")
        print("   Continuing without HandPD (Kaggle datasets are sufficient)")
else:
    print("   ✅ HandPD already downloaded")

# ═══════════════════════════════════════════════════════════
# Verify all datasets
# ═══════════════════════════════════════════════════════════
print("\n📂 Dataset availability:")
active_datasets = {}
for key, info in DATASET_REGISTRY.items():
    path = info["path"]
    exists = os.path.exists(path)
    status = "✅ FOUND" if exists else "❌ Not found"
    print(f"   {status} — {info['name']}")
    if exists:
        # Count image files
        img_count = 0
        for root, dirs, files in os.walk(path):
            img_count += sum(1 for f in files if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'})
        print(f"          → {img_count} image files at {path}")
        if img_count > 0:
            active_datasets[key] = info
        else:
            print(f"          → Skipping (no images)")

print(f"\n🎯 Active datasets: {len(active_datasets)} / {len(DATASET_REGISTRY)}")
if len(active_datasets) == 0:
    raise RuntimeError(
        "No datasets found! Add at least 'kmader/parkinsons-drawings' "
        "to this notebook via Add Data in the Kaggle sidebar."
    )

## 2️⃣ Smart Multi-Dataset Scanner

The scanner auto-detects drawing type and class from **any folder structure**:
- Folder name contains `spiral`/`wave`/`meander` → drawing type
- Folder name contains `healthy`/`control`/`normal` → class = healthy
- Folder name contains `parkinson`/`patient`/`pd` → class = PD
- `train`/`test` → split assignment

### Supported structures (all handled automatically):
```
kmader:        spiral/training/healthy/*.png
alihussain:    spiral/healthy/*.png  OR  spiral_healthy/*.png
vasantha:      healthy/*.png  OR  parkinson/*.png
handpd:        HandPD/spiral/control/*.png, HandPD/spiral/patient/*.png
newhandpd:     NewHandPD/spiral/control/*.png, NewHandPD/spiral/patient/*.png
```

Datasets are **deduplicated by image hash** to avoid training on duplicate images.

In [ ]:
# ============================================================
# CELL 2: Smart Multi-Dataset Scanner (with deduplication)
# ============================================================
import hashlib

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif'}

# ═══════════════════════════════════════════════════════════
# Smart folder-name parser — works across ALL dataset formats
# ═══════════════════════════════════════════════════════════
def parse_path_metadata(full_path, dataset_root):
    """
    Parse drawing_type, split, and class from folder structure.
    Handles: kmader, alihussain, vasantha, HandPD, NewHandPD.
    """
    relative = full_path.replace(dataset_root, "").strip(os.sep).lower()
    parts = relative.replace("\\", "/").split("/")
    
    drawing_type = None
    split_name = None
    class_name = None
    
    for part in parts:
        # Drawing type detection
        if 'spiral' in part:
            drawing_type = 'spiral'
        elif 'wave' in part:
            drawing_type = 'wave'
        elif 'meander' in part:
            drawing_type = 'meander'  # HandPD uses meander (similar to wave)
        
        # Split detection
        if part in ('training', 'train'):
            split_name = 'train'
        elif part in ('testing', 'test'):
            split_name = 'test'
        
        # Class detection (broad matching)
        if part in ('healthy', 'health', 'control', 'controls', 'normal', 'h'):
            class_name = 'healthy'
        elif part in ('parkinson', 'parkinsons', 'patient', 'patients', 'pd', 'p'):
            class_name = 'parkinson'
    
    # Some datasets embed class in filename (e.g., "H_001.png" or "P_001.png")
    if class_name is None:
        fname = os.path.basename(full_path).lower()
        if fname.startswith(('h_', 'h0', 'healthy', 'control')):
            class_name = 'healthy'
        elif fname.startswith(('p_', 'p0', 'parkinson', 'patient', 'pd')):
            class_name = 'parkinson'
    
    # Treat meander as wave (both test motor fluidity)
    if drawing_type == 'meander':
        drawing_type = 'wave'
    
    # Default split to train if not specified
    if split_name is None:
        split_name = 'train'
    
    return drawing_type, split_name, class_name


def compute_image_hash(filepath, hash_size=8):
    """Fast perceptual hash for deduplication (uses file content hash)."""
    try:
        with open(filepath, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None


# ═══════════════════════════════════════════════════════════
# Scan ALL active datasets
# ═══════════════════════════════════════════════════════════
all_images = []
seen_hashes = set()
duplicate_count = 0

for ds_key, ds_info in active_datasets.items():
    ds_path = ds_info["path"]
    ds_name = ds_info["name"]
    ds_count = 0
    ds_skipped = 0
    
    for root, dirs, files in os.walk(ds_path):
        for f in files:
            ext = Path(f).suffix.lower()
            if ext not in IMAGE_EXTS:
                continue
            
            full_path = os.path.join(root, f)
            
            # Deduplication via content hash
            img_hash = compute_image_hash(full_path)
            if img_hash and img_hash in seen_hashes:
                duplicate_count += 1
                ds_skipped += 1
                continue
            if img_hash:
                seen_hashes.add(img_hash)
            
            drawing_type, split_name, class_name = parse_path_metadata(full_path, ds_path)
            
            # Skip images where we can't determine the class
            if class_name is None:
                ds_skipped += 1
                continue
            
            # If drawing_type unknown, try to infer from dataset key
            if drawing_type is None:
                if 'spiral' in ds_key:
                    drawing_type = 'spiral'
                else:
                    drawing_type = 'spiral'  # default assumption
            
            all_images.append({
                'path': full_path,
                'drawing_type': drawing_type,
                'split': split_name,
                'class_name': class_name,
                'label': 0 if class_name == 'healthy' else 1,
                'filename': f,
                'dataset': ds_key,
                'dataset_name': ds_name,
            })
            ds_count += 1
    
    print(f"   📦 {ds_name}: {ds_count} valid images ({ds_skipped} skipped/duplicates)")

df_all = pd.DataFrame(all_images)

# ═══════════════════════════════════════════════════════════
# Comprehensive Summary
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("📊 COMBINED MULTI-DATASET SCAN RESULTS")
print("=" * 70)
print(f"\n   Total unique images: {len(df_all)}")
print(f"   Duplicates removed:  {duplicate_count}")
print(f"   Datasets loaded:     {df_all['dataset'].nunique()}")

print(f"\n   📋 By Dataset:")
for ds, count in df_all['dataset_name'].value_counts().items():
    print(f"     {ds}: {count}")

print(f"\n   📋 By Drawing Type:")
for dt, count in df_all['drawing_type'].value_counts().items():
    print(f"     {dt}: {count}")

print(f"\n   📋 By Class:")
for cls, count in df_all['class_name'].value_counts().items():
    print(f"     {cls}: {count}")

balance = df_all['class_name'].value_counts()
ratio = balance.min() / balance.max()
print(f"\n   ⚖️  Class balance: {ratio:.2f} (1.0 = perfect)")
if ratio < 0.7:
    print(f"   ⚠️  Imbalanced! WeightedRandomSampler will fix this.")

print(f"\n   📋 Detailed Breakdown:")
breakdown = df_all.groupby(['dataset', 'drawing_type', 'class_name']).size().reset_index(name='count')
for _, row in breakdown.iterrows():
    print(f"     {row['dataset']:20s} | {row['drawing_type']:8s} | {row['class_name']:10s} | {row['count']:4d}")

# Separate spiral and wave DataFrames (for later analysis)
df_spiral = df_all[df_all['drawing_type'] == 'spiral'].copy()
df_wave = df_all[df_all['drawing_type'] == 'wave'].copy()

print(f"\n   🌀 Spirals: {len(df_spiral)} | 🌊 Waves: {len(df_wave)}")
print(f"\n✅ Multi-dataset loading complete — {len(df_all)} unique images ready!")

## 3️⃣ Dataset Visualization — Healthy vs Parkinson's Drawings

Visual comparison of drawing quality differences between healthy controls and PD patients.

In [ ]:
# ============================================================
# CELL 3: Visualize Sample Drawings
# ============================================================

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
fig.suptitle('Healthy vs Parkinson\'s — Spiral & Wave Drawings', fontsize=16, fontweight='bold')

# Row labels
axes[0, 0].set_ylabel('SPIRAL', fontsize=14, fontweight='bold', rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('WAVE', fontsize=14, fontweight='bold', rotation=0, labelpad=60, va='center')

for row_idx, drawing_type in enumerate(['spiral', 'wave']):
    df_type = df_all[df_all['drawing_type'] == drawing_type]
    
    # 3 healthy + 3 parkinson
    for col_idx, class_name in enumerate(['healthy', 'parkinson']):
        subset = df_type[df_type['class_name'] == class_name]
        samples = subset.sample(min(3, len(subset)), random_state=SEED)
        
        for i, (_, row) in enumerate(samples.iterrows()):
            ax_col = col_idx * 3 + i
            try:
                img = Image.open(row['path']).convert('RGB')
                axes[row_idx, ax_col].imshow(img)
            except Exception as e:
                axes[row_idx, ax_col].text(0.5, 0.5, 'Error', ha='center', va='center')
            
            color = 'green' if class_name == 'healthy' else 'red'
            axes[row_idx, ax_col].set_title(
                f"{'✅ Healthy' if class_name == 'healthy' else '🔴 PD'}",
                fontsize=10, color=color, fontweight='bold'
            )
            axes[row_idx, ax_col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'pd_drawing_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print("← Healthy Controls ───────── Parkinson's Disease →")

# ═══════════════════════════════════════════════════════════
# Image size distribution
# ═══════════════════════════════════════════════════════════
print(f"\n📐 Image Size Distribution:")
sizes = []
for _, row in df_all.sample(min(50, len(df_all))).iterrows():
    try:
        img = Image.open(row['path'])
        sizes.append(img.size)
    except:
        pass

widths = [s[0] for s in sizes]
heights = [s[1] for s in sizes]
print(f"   Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
print(f"   Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")

## 4️⃣ Heavy Data Augmentation Strategy

### Why augmentation?
Even with ~800+ images from multiple sources, the dataset is **still relatively small** for deep learning.
Heavy augmentation prevents overfitting and simulates real-world variability:
- **Rotation ±30°** — paper orientation variation
- **Elastic deformation** — simulates natural pen stroke variation
- **Gaussian noise** — scanner/camera quality variation
- **Brightness/contrast jitter** — lighting conditions from different datasets
- **Perspective distortion** — camera/scanner angle differences
- **Random erasing** — small occlusions
- **Horizontal + Vertical flip** — drawings are symmetric (unlike clock drawings!)
- **Affine transformations** — scale/shift/shear

With 5x oversampling: **~800 images → ~4,000 effective training samples per epoch**

In [ ]:
# ============================================================
# CELL 4: Custom Dataset & Augmentation Pipeline
# ============================================================

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ═══════════════════════════════════════════════════════════
# HEAVY augmentation — 12 transforms for diverse training
# ═══════════════════════════════════════════════════════════
train_transforms = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    T.RandomCrop(IMG_SIZE),
    T.RandomRotation(30),                                  # ±30° — generous rotation
    T.RandomHorizontalFlip(p=0.5),                         # OK for spirals/waves (symmetric)
    T.RandomVerticalFlip(p=0.3),                           # Also OK
    T.RandomPerspective(distortion_scale=0.15, p=0.4),     # Camera/scanner angle
    T.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),                            # Position shift
        scale=(0.85, 1.15),                                # Scale variation
        shear=(-10, 10),                                   # Shear deformation
    ),
    T.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.2,
        hue=0.05,
    ),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),      # Pen/scan blur
    T.RandomGrayscale(p=0.3),                              # Many originals are grayscale
    T.RandomInvert(p=0.1),                                 # Some scans have inverted colors
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.15)),           # Random occlusions
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ═══════════════════════════════════════════════════════════
# Custom Dataset — supports both spiral and wave drawings
# ═══════════════════════════════════════════════════════════
class ParkinsonsDrawingDataset(Dataset):
    """
    Parkinson's spiral/wave drawing dataset.
    
    Labels: 0 = Healthy, 1 = Parkinson's Disease
    """
    
    def __init__(self, dataframe, transform=None, oversample_factor=1):
        """
        Args:
            dataframe: DataFrame with 'path', 'label' columns
            transform: Image transforms
            oversample_factor: Repeat dataset N times per epoch (for small datasets)
        """
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform
        self.oversample_factor = oversample_factor
    
    def __len__(self):
        return len(self.data) * self.oversample_factor
    
    def __getitem__(self, idx):
        # Wrap around for oversampling
        real_idx = idx % len(self.data)
        row = self.data.iloc[real_idx]
        
        # Load image
        image = Image.open(row['path']).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, row['label']

CLASS_NAMES = {0: 'Healthy', 1: "Parkinson's"}
print(f"✅ Dataset and transforms defined")
print(f"   Classes: {CLASS_NAMES}")
print(f"   Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Train augmentation: 12 transforms (heavy for small dataset)")

In [ ]:
# ============================================================
# CELL 5: Train/Val/Test Split & DataLoaders
# ============================================================

# ═══════════════════════════════════════════════════════════
# Strategy: Train on ALL drawing types together (spiral + wave/meander)
# The model learns general PD motor impairment patterns
# Multi-dataset images are all merged — deduplication already done
# ═══════════════════════════════════════════════════════════

# For datasets with pre-defined splits, respect them.
# For others (HandPD, NewHandPD, etc.), everything is 'train' by default.
df_presplit_test = df_all[df_all['split'] == 'test'].copy()
df_presplit_train = df_all[df_all['split'] == 'train'].copy()

# If some datasets have test splits, use them; otherwise split manually
if len(df_presplit_test) >= 10:
    # Use existing test set + split training into train/val
    df_train_full = df_presplit_train.copy()
    df_test_orig = df_presplit_test.copy()
else:
    # No meaningful test split — create one from all data
    df_train_full, df_test_orig = train_test_split(
        df_all, test_size=0.15, random_state=SEED,
        stratify=df_all['class_name']
    )

# Create validation set from training data
# Stratify by class (and drawing_type if enough samples per group)
try:
    stratify_col = df_train_full[['class_name', 'drawing_type']].apply(tuple, axis=1)
    # Check minimum group size
    min_group = stratify_col.value_counts().min()
    if min_group < 2:
        raise ValueError("Group too small for combined stratification")
    df_train, df_val = train_test_split(
        df_train_full, test_size=0.15, random_state=SEED,
        stratify=stratify_col
    )
except (ValueError, KeyError):
    # Fall back to class-only stratification
    df_train, df_val = train_test_split(
        df_train_full, test_size=0.15, random_state=SEED,
        stratify=df_train_full['class_name']
    )

print("=" * 70)
print("📊 SPLIT SUMMARY (multi-dataset)")
print("=" * 70)
print(f"   Train: {len(df_train)} images")
print(f"   Val:   {len(df_val)} images")
print(f"   Test:  {len(df_test_orig)} images")
print(f"   Total: {len(df_train) + len(df_val) + len(df_test_orig)} images")

for split_name, df_split in [("Train", df_train), ("Val", df_val), ("Test", df_test_orig)]:
    print(f"\n   {split_name} breakdown:")
    for _, row in df_split.groupby(['drawing_type', 'class_name']).size().reset_index(name='n').iterrows():
        print(f"     {row['drawing_type']:8s} | {row['class_name']:10s} | {row['n']}")
    # Show dataset sources
    print(f"     Sources: {', '.join(df_split['dataset'].unique())}")

# ═══════════════════════════════════════════════════════════
# Weighted Sampler for balanced training
# ═══════════════════════════════════════════════════════════
class_counts = df_train['label'].value_counts().sort_index()
class_weights = 1.0 / class_counts.values
sample_weights = df_train['label'].map(dict(zip(class_counts.index, class_weights))).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"\n   Class weights: Healthy={class_weights[0]:.3f}, PD={class_weights[1]:.3f}")

# ═══════════════════════════════════════════════════════════
# Oversampling factor — adaptive based on dataset size
# Small (<200): 10x  |  Medium (200-500): 5x  |  Large (500+): 3x
# ═══════════════════════════════════════════════════════════
n_train = len(df_train)
if n_train < 200:
    OVERSAMPLE = 10
elif n_train < 500:
    OVERSAMPLE = 5
else:
    OVERSAMPLE = 3

BATCH_SIZE = 32 if n_train >= 300 else 16

train_dataset = ParkinsonsDrawingDataset(df_train, transform=train_transforms, oversample_factor=OVERSAMPLE)
val_dataset = ParkinsonsDrawingDataset(df_val, transform=val_transforms)
test_dataset = ParkinsonsDrawingDataset(df_test_orig, transform=val_transforms)

# WeightedRandomSampler for oversampled dataset
oversample_weights = np.tile(sample_weights, OVERSAMPLE)
oversample_sampler = WeightedRandomSampler(
    weights=oversample_weights,
    num_samples=len(oversample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=oversample_sampler,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"\n   Dataset size: {n_train} → Oversampling: {OVERSAMPLE}x → {len(train_dataset)} effective samples")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches/epoch: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

# ═══════════════════════════════════════════════════════════
# Preview augmented samples
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Augmented Training Samples (same image → different augmentations)', fontsize=14, fontweight='bold')

for i in range(12):
    row, col = i // 6, i % 6
    img, label = train_dataset[i]
    img_show = img.permute(1, 2, 0).numpy()
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_show = np.clip(img_show, 0, 1)
    axes[row, col].imshow(img_show)
    axes[row, col].set_title(CLASS_NAMES[label], fontsize=9,
                              color='green' if label == 0 else 'red')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print(f"✅ DataLoaders ready with {OVERSAMPLE}x oversampling!")

## 5️⃣ Model Architecture — MotorNet (EfficientNet-B0 + PD Head)

### Why EfficientNet-B0 for small datasets?
- **ImageNet pretraining** transfers low-level features (edges, textures, curves) directly useful for tremor detection
- **5.3M parameters** but only ~800K trainable after freezing — reduces overfitting risk
- **Compound scaling** — optimal depth/width/resolution balance
- **GradCAM-ready** — conv_head layer gives clinically interpretable saliency maps

### Architecture:
```
EfficientNet-B0 (freeze blocks 0-4, fine-tune blocks 5-6)
    ↓ 1280-d feature vector
Dropout(0.5) → Linear(1280, 256) → ReLU → BatchNorm → Dropout(0.3)
    ↓
Linear(256, 64) → ReLU → BatchNorm
    ↓
Linear(64, 2) ← Healthy vs PD
    ↓ (parallel branch)
Linear(2, 16) → ReLU → Linear(16, 1) → Sigmoid ← PD risk [0,1]
```

### Higher dropout for small dataset:
- `Dropout(0.5)` at the top layer — aggressive regularization
- Smaller hidden dimensions (256→64 instead of 512→128)

In [ ]:
# ============================================================
# CELL 6: Model Architecture — MotorNet
# ============================================================

class MotorNet(nn.Module):
    """
    Parkinson's Disease motor drawing classifier based on EfficientNet-B0.
    
    Trained on spiral + wave drawings from Zham et al. (2017).
    Outputs:
    - class_logits: [Healthy, PD] binary classification
    - pd_risk: Continuous PD risk score [0, 1]
    
    GradCAM compatible — target_layer = self.backbone.conv_head
    """
    
    def __init__(self, num_classes=2, pretrained=True, dropout=0.5):
        super().__init__()
        
        # ── EfficientNet-B0 backbone ──
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            num_classes=0,       # Remove classifier head
            global_pool='avg'    # → 1280-d feature vector
        )
        
        # Freeze bottom layers (blocks 0-4)
        # Fine-tune blocks 5, 6, conv_head, bn2
        for name, param in self.backbone.named_parameters():
            if any(x in name for x in ['blocks.5', 'blocks.6', 'conv_head', 'bn2']):
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.backbone.parameters() if not p.requires_grad)
        
        # ── Classification head (smaller for tiny dataset) ──
        feature_dim = self.backbone.num_features  # 1280
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),                    # 0.5 — aggressive for small data
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.6),              # 0.3
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes),
        )
        
        # ── PD Risk head (regression) ──
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        
        head_params = sum(p.numel() for p in self.classifier.parameters()) + \
                      sum(p.numel() for p in self.risk_head.parameters())
        
        print(f"🏗️  MotorNet Architecture:")
        print(f"   Backbone: EfficientNet-B0 ({frozen:,} frozen + {trainable:,} trainable)")
        print(f"   Head: {head_params:,} parameters")
        print(f"   Total trainable: {trainable + head_params:,}")
        print(f"   Dropout: {dropout} (aggressive for small dataset)")
        print(f"   Output: {num_classes} classes + PD risk score")
    
    def forward(self, x):
        features = self.backbone(x)          # [B, 1280]
        logits = self.classifier(features)   # [B, 2]
        
        # PD risk from class probabilities
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)         # [B, 1]
        
        return logits, risk.squeeze(-1)
    
    def get_features(self, x):
        """Get backbone features for GradCAM."""
        return self.backbone(x)


# ═══════════════════════════════════════════════════════════
# Instantiate model
# ═══════════════════════════════════════════════════════════
NUM_CLASSES = 2
model = MotorNet(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5).to(device)

# Verify with dummy input
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
logits, risk = model(dummy)
print(f"\n✅ Forward pass test:")
print(f"   Input:  {dummy.shape}")
print(f"   Logits: {logits.shape} → {CLASS_NAMES}")
print(f"   Risk:   {risk.shape} (PD risk scores)")
print(f"   Risk values: {risk.detach().cpu().numpy()}")

## 6️⃣ Training Configuration

### Strategy for Small Dataset:
- **Label Smoothing (0.15)** — stronger than CDT (0.1) to prevent overconfident predictions
- **Weight Decay (5e-4)** — stronger L2 regularization
- **Cosine Annealing with Warm Restarts** — escape local minima
- **Early Stopping (patience=15)** — generous patience due to small/noisy validation set
- **Gradient clipping** — stability for small batches

In [ ]:
# ============================================================
# CELL 7: Training Loop with Early Stopping
# ============================================================

# ═══════════════════════════════════════════════════════════
# Hyperparameters (tuned for small dataset)
# ═══════════════════════════════════════════════════════════
NUM_EPOCHS = 60
LEARNING_RATE = 5e-4        # Lower than CDT — careful with small data
WEIGHT_DECAY = 5e-4         # Stronger L2 regularization
LABEL_SMOOTHING = 0.15      # Stronger smoothing for overfit prevention
PATIENCE = 15               # Generous patience (small val set = noisy metrics)

# ═══════════════════════════════════════════════════════════
# Loss, Optimizer, Scheduler
# ═══════════════════════════════════════════════════════════
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Differential learning rates
backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
head_params = list(model.classifier.parameters()) + list(model.risk_head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE / 10},   # Backbone: very slow
    {'params': head_params, 'lr': LEARNING_RATE},              # Head: normal
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

# Mixed precision
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

# ═══════════════════════════════════════════════════════════
# Training functions
# ═══════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0
    all_preds, all_labels = [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                logits, risk = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, risk = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        logits, risk = model(images)
        loss = criterion(logits, labels)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)


# ═══════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("🚀 STARTING TRAINING — MotorNet (Spiral + Wave)")
print("=" * 60)
print(f"   Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"   Oversample: {OVERSAMPLE}x | Label Smoothing: {LABEL_SMOOTHING}")
print(f"   Early stopping patience: {PATIENCE}")
print()

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_f1': [], 'val_f1': [],
    'lr': []
}

best_val_f1 = 0
patience_counter = 0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    val_loss, val_acc, val_f1, _, _, _ = validate(
        model, val_loader, criterion, device
    )
    
    # Scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # Print
    print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} │ "
          f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f} │ "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f} │ "
          f"LR: {current_lr:.2e}", end="")
    
    # Save best
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
            'img_size': IMG_SIZE,
            'class_names': CLASS_NAMES,
        }, os.path.join(MODEL_DIR, 'motor_best.pt'))
        
        print(f" ★ BEST", end="")
    else:
        patience_counter += 1
    
    print()
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Best epoch: {best_epoch} with val F1 = {best_val_f1:.4f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 8: Training Curves
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
axes[2].plot(history['val_f1'], label='Validation', linewidth=2)
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[2].set_title('Weighted F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Learning Rate
axes[3].plot(history['lr'], linewidth=2, color='purple')
axes[3].set_title('Learning Rate', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Epoch')
axes[3].set_yscale('log')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Test Set Evaluation

Load the best checkpoint and evaluate on the held-out test set.
Since this is a binary task, we get:
- **Accuracy**, **F1**, **Sensitivity** (true positive rate for PD), **Specificity** (true negative rate)
- **ROC-AUC** — area under the receiver operating characteristic curve
- **Confusion matrix** — shows false positives vs false negatives

In [ ]:
# ============================================================
# CELL 9: Test Set Evaluation
# ============================================================

# Load best model
checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} (val F1={checkpoint['val_f1']:.4f})")

# Evaluate on test set
test_loss, test_acc, test_f1, test_preds, test_labels, test_probs = validate(
    model, test_loader, criterion, device
)

print(f"\n{'='*60}")
print(f"📊 TEST SET RESULTS")
print(f"{'='*60}")
print(f"   Accuracy:      {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"   F1 (weighted): {test_f1:.4f}")
print(f"   Loss:          {test_loss:.4f}")

# Per-class report
class_names_list = ['Healthy', "Parkinson's"]
print(f"\n📋 Classification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names_list, digits=3))

# ROC-AUC
try:
    auc = roc_auc_score(test_labels, test_probs[:, 1])
    print(f"   ROC-AUC: {auc:.4f}")
except Exception as e:
    print(f"   ROC-AUC: Could not compute ({e})")

# Sensitivity & Specificity
cm = confusion_matrix(test_labels, test_preds)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    print(f"\n   Clinical Metrics:")
    print(f"   Sensitivity (PD detection): {sensitivity:.3f} ({sensitivity*100:.1f}%)")
    print(f"   Specificity (Healthy correct): {specificity:.3f} ({specificity*100:.1f}%)")
    print(f"   PPV (Positive Predictive Value): {ppv:.3f}")
    print(f"   NPV (Negative Predictive Value): {npv:.3f}")

# ═══════════════════════════════════════════════════════════
# Confusion Matrix
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[0],
            annot_kws={"size": 18})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[1],
            annot_kws={"size": 18})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════
# Separate evaluation: Spiral-only vs Wave-only
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📊 SEPARATE EVALUATION — Spiral vs Wave")
print(f"{'='*60}")

for drawing_type in ['spiral', 'wave']:
    df_type_test = df_test_orig[df_test_orig['drawing_type'] == drawing_type]
    if len(df_type_test) == 0:
        continue
    
    type_dataset = ParkinsonsDrawingDataset(df_type_test, transform=val_transforms)
    type_loader = DataLoader(type_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    _, type_acc, type_f1, type_preds, type_labels, type_probs = validate(
        model, type_loader, criterion, device
    )
    
    type_auc = roc_auc_score(type_labels, type_probs[:, 1]) if len(np.unique(type_labels)) > 1 else 0
    
    print(f"\n   {drawing_type.upper()}:")
    print(f"     Accuracy: {type_acc:.3f} | F1: {type_f1:.3f} | AUC: {type_auc:.3f}")
    print(f"     ({len(df_type_test)} test images)")

## 8️⃣ GradCAM — Explainable AI for Motor Drawings

### What GradCAM reveals for PD drawings:
- **Tremor hotspots** — jagged/wavy regions where hand shook
- **Micrographia** — areas where spiral gets progressively tighter
- **Irregularity zones** — breaks in smooth curvature (rigidity)
- **Pen pressure variation** — darker/lighter regions from uneven pressure

This is critical for NeuroVerse's XAI module — doctors can see **exactly where** the model detected PD signs in the patient's drawing.

In [ ]:
# ============================================================
# CELL 10: GradCAM Implementation
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class MotorModelWrapper(nn.Module):
    """Wrapper that returns only logits (GradCAM needs single output)."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

wrapped_model = MotorModelWrapper(model)

# Target layer — last convolutional layer of EfficientNet-B0
target_layer = model.backbone.conv_head

# Initialize GradCAM
cam = GradCAM(model=wrapped_model, target_layers=[target_layer])

# ═══════════════════════════════════════════════════════════
# Generate GradCAM for a single image
# ═══════════════════════════════════════════════════════════
def generate_gradcam(image_path, true_label=None):
    """Generate GradCAM for a drawing image."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Prediction
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        risk_val = risk.item()
    
    # GradCAM for predicted class
    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
    
    # Overlay
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    return {
        'original': img_np,
        'gradcam': cam_image,
        'heatmap': grayscale_cam,
        'pred_class': pred_class,
        'true_label': true_label,
        'probs': probs,
        'risk': risk_val
    }


# ═══════════════════════════════════════════════════════════
# Visualize: Healthy vs PD — Spiral + Wave GradCAMs
# ═══════════════════════════════════════════════════════════
print("🔍 Generating GradCAM visualizations...")

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('GradCAM: Healthy vs PD — What the Model Sees\n'
             '(Red = High Attention for PD detection | Blue = Low Attention)',
             fontsize=16, fontweight='bold')

columns = [
    ('spiral', 'healthy', 'Spiral — Healthy'),
    ('spiral', 'parkinson', 'Spiral — PD'),
    ('wave', 'healthy', 'Wave — Healthy'),
    ('wave', 'parkinson', 'Wave — PD'),
]

for col_idx, (drawing_type, class_name, title) in enumerate(columns):
    subset = df_test_orig[
        (df_test_orig['drawing_type'] == drawing_type) &
        (df_test_orig['class_name'] == class_name)
    ]
    
    if len(subset) == 0:
        subset = df_all[
            (df_all['drawing_type'] == drawing_type) &
            (df_all['class_name'] == class_name)
        ]
    
    if len(subset) == 0:
        for row in range(3):
            axes[row, col_idx].axis('off')
        continue
    
    sample = subset.sample(1, random_state=SEED + col_idx).iloc[0]
    result = generate_gradcam(sample['path'], true_label=sample['label'])
    
    # Row 0: Original
    axes[0, col_idx].imshow(result['original'])
    axes[0, col_idx].set_title(title, fontsize=11, fontweight='bold',
                                color='green' if class_name == 'healthy' else 'red')
    axes[0, col_idx].axis('off')
    
    # Row 1: GradCAM overlay
    axes[1, col_idx].imshow(result['gradcam'])
    pred_name = CLASS_NAMES[result['pred_class']]
    correct = result['pred_class'] == sample['label']
    axes[1, col_idx].set_title(
        f"Pred: {pred_name} ({'✓' if correct else '✗'})\n"
        f"PD Risk: {result['risk']:.2f}",
        fontsize=9, color='green' if correct else 'red'
    )
    axes[1, col_idx].axis('off')
    
    # Row 2: Heatmap only
    axes[2, col_idx].imshow(result['heatmap'], cmap='jet')
    conf = result['probs'][result['pred_class']]
    axes[2, col_idx].set_title(f"Confidence: {conf:.1%}", fontsize=9)
    axes[2, col_idx].axis('off')

# Row labels
for i, label in enumerate(['Original', 'GradCAM', 'Heatmap']):
    axes[i, 0].set_ylabel(label, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_gradcam_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ GradCAM visualizations saved!")

In [ ]:
# ============================================================
# CELL 11: GradCAM — All Test Samples + Misclassified Analysis
# ============================================================

# Find correct and misclassified examples
correct_mask = test_preds == test_labels
correct_indices = np.where(correct_mask)[0]
wrong_indices = np.where(~correct_mask)[0]

print(f"   Correct: {len(correct_indices)} | Misclassified: {len(wrong_indices)}")
print(f"   Test Accuracy: {len(correct_indices)/(len(correct_indices)+len(wrong_indices)):.1%}")

if len(wrong_indices) > 0:
    n_show = min(6, len(wrong_indices))
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    fig.suptitle('GradCAM on MISCLASSIFIED Samples\n(What confused the model?)',
                 fontsize=14, fontweight='bold')
    
    if n_show == 1:
        axes = axes.reshape(2, 1)
    
    for i, idx in enumerate(wrong_indices[:n_show]):
        row = df_test_orig.iloc[idx]
        result = generate_gradcam(row['path'], true_label=int(test_labels[idx]))
        
        # Original
        axes[0, i].imshow(result['original'])
        axes[0, i].set_title(
            f"True: {CLASS_NAMES[int(test_labels[idx])]}\n"
            f"Pred: {CLASS_NAMES[int(test_preds[idx])]}",
            fontsize=9, color='red', fontweight='bold'
        )
        axes[0, i].axis('off')
        
        # GradCAM
        axes[1, i].imshow(result['gradcam'])
        axes[1, i].set_title(
            f"{row['drawing_type'].upper()}\n"
            f"Risk: {result['risk']:.2f} | Conf: {result['probs'][result['pred_class']]:.1%}",
            fontsize=8
        )
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'motor_misclassified_gradcam.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("🎉 No misclassifications on test set!")

## 9️⃣ Full Fine-tuning (Unfreeze All Layers)

After the head is well-trained, unfreeze ALL backbone layers for a few more epochs at very low LR.
This typically adds **2-5% accuracy** on small datasets — the backbone layers adapt to the specific drawing domain.

In [ ]:
# ============================================================
# CELL 12: Full Fine-tuning (Optional)
# ============================================================

FULL_FINETUNE = True  # Set False to skip

if FULL_FINETUNE:
    print("🔓 Unfreezing ALL backbone layers for full fine-tuning...")
    
    for param in model.parameters():
        param.requires_grad = True
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable parameters: {total_params:,}")
    
    FT_LR = 5e-6       # Very conservative for small dataset
    FT_EPOCHS = 20
    FT_PATIENCE = 10
    
    ft_optimizer = optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
    ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-8)
    
    print(f"   LR: {FT_LR} | Epochs: {FT_EPOCHS} | Patience: {FT_PATIENCE}")
    print()
    
    ft_patience_counter = 0
    
    for epoch in range(FT_EPOCHS):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, ft_optimizer, scaler, device
        )
        val_loss, val_acc, val_f1, _, _, _ = validate(model, val_loader, criterion, device)
        ft_scheduler.step()
        
        print(f"  FT Epoch {epoch+1:2d}/{FT_EPOCHS} │ "
              f"Train: acc={train_acc:.3f} f1={train_f1:.3f} │ "
              f"Val: acc={val_acc:.3f} f1={val_f1:.3f}", end="")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ft_patience_counter = 0
            torch.save({
                'epoch': NUM_EPOCHS + epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': ft_optimizer.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
                'num_classes': NUM_CLASSES,
                'img_size': IMG_SIZE,
                'class_names': CLASS_NAMES,
                'full_finetuned': True,
            }, os.path.join(MODEL_DIR, 'motor_best.pt'))
            print(f" ★ NEW BEST (f1={val_f1:.4f})")
        else:
            ft_patience_counter += 1
            print()
        
        if ft_patience_counter >= FT_PATIENCE:
            print(f"\n⏹️  Fine-tuning stopped (no improvement for {FT_PATIENCE} epochs)")
            break
    
    print(f"\n✅ Fine-tuning complete! Best F1: {best_val_f1:.4f}")
    
    # Reload best
    checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("⏭️  Skipping full fine-tuning")

## 🔟 Model Export for NeuroVerse Backend

Export the trained model in formats for deployment:
1. **`motor_spiral_model.pt`** — Full PyTorch model for backend (`app/ml/models/`)
2. **`motor_spiral_model_mobile.pt`** — TorchScript for mobile
3. **`motor_config.json`** — Class mapping + UPDRS thresholds for `fusion_service.py`

### How it maps to UPDRS (MDS-UPDRS 3.15-3.18):
| Prediction | UPDRS Tremor | Clinical Meaning |
|-----------|-------------|-----------------|
| Healthy (conf > 0.90) | 0 | Normal |
| Healthy (conf 0.70-0.90) | 1 | Slight tremor possible |
| PD (conf 0.50-0.70) | 2 | Mild tremor |
| PD (conf 0.70-0.85) | 3 | Moderate tremor |
| PD (conf > 0.85) | 4 | Severe tremor |

In [ ]:
# ============================================================
# CELL 13: Model Export
# ============================================================

EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Full PyTorch model (for backend)
# ═══════════════════════════════════════════════════════════
export_path = os.path.join(EXPORT_DIR, "motor_spiral_model.pt")

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'architecture': 'efficientnet_b0',
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
        'dropout': 0.5,
    },
    'class_mapping': {
        'class_names': CLASS_NAMES,
        'label_to_class': {0: 'healthy', 1: 'parkinson'},
    },
    'normalization': {
        'mean': IMAGENET_MEAN,
        'std': IMAGENET_STD,
    },
    'metrics': {
        'test_accuracy': float(test_acc),
        'test_f1': float(test_f1),
        'best_val_f1': float(best_val_f1),
    },
    'clinical_thresholds': {
        # Maps model confidence → UPDRS tremor score (3.15-3.18)
        'updrs_mapping': {
            'healthy_high_conf':   {'min_conf': 0.90, 'updrs_tremor': 0, 'label': 'Normal'},
            'healthy_moderate':    {'min_conf': 0.70, 'updrs_tremor': 1, 'label': 'Slight'},
            'pd_low_conf':         {'min_conf': 0.50, 'updrs_tremor': 2, 'label': 'Mild tremor'},
            'pd_moderate_conf':    {'min_conf': 0.70, 'updrs_tremor': 3, 'label': 'Moderate tremor'},
            'pd_high_conf':        {'min_conf': 0.85, 'updrs_tremor': 4, 'label': 'Severe tremor'},
        },
        # Risk mapping for fusion_service.py
        'pd_risk_by_prediction': {
            'healthy': {'pd_risk': 0.05, 'ad_risk': 0.02},
            'parkinson': {'pd_risk': 0.65, 'ad_risk': 0.05},
        },
        # Tremor frequency range (PD-characteristic)
        'pd_tremor_frequency_hz': [4.0, 6.0],
        # Tapping threshold from fusion_service.py
        'tapping_pd_threshold': 3.0,
    },
    'dataset_info': {
        'name': 'parkinsons-drawings (Zham et al., 2017)',
        'source': 'Kaggle: kmader/parkinsons-drawings',
        'drawing_types': ['spiral', 'wave'],
        'paper': 'doi:10.3389/fneur.2017.00435',
    }
}, export_path)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"✅ Full model exported: {export_path} ({size_mb:.1f} MB)")

# ═══════════════════════════════════════════════════════════
# 2. TorchScript model (mobile / ONNX)
# ═══════════════════════════════════════════════════════════
try:
    model.eval()
    
    class MotorInference(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.backbone = model.backbone
            self.classifier = model.classifier
        
        def forward(self, x):
            features = self.backbone(x)
            logits = self.classifier(features)
            return logits
    
    inference_model = MotorInference(model)
    scripted = torch.jit.trace(inference_model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device))
    script_path = os.path.join(EXPORT_DIR, "motor_spiral_model_mobile.pt")
    scripted.save(script_path)
    size_mb_s = os.path.getsize(script_path) / (1024 * 1024)
    print(f"✅ TorchScript model: {script_path} ({size_mb_s:.1f} MB)")
except Exception as e:
    print(f"⚠️  TorchScript export failed (non-critical): {e}")

# ═══════════════════════════════════════════════════════════
# 3. Config JSON
# ═══════════════════════════════════════════════════════════
config = {
    "model_name": "MotorNet",
    "version": "1.0",
    "architecture": "efficientnet_b0",
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "class_names": CLASS_NAMES,
    "drawing_types": ["spiral", "wave"],
    "updrs_mapping": {
        "0": {"pd_risk": 0.05, "updrs_tremor": 0, "label": "Normal — No tremor"},
        "1": {"pd_risk": 0.65, "updrs_tremor": 2, "label": "PD detected — Tremor present"},
    },
    "confidence_to_updrs": [
        {"pred": "healthy", "conf_min": 0.90, "updrs": 0, "severity": "Normal"},
        {"pred": "healthy", "conf_min": 0.70, "updrs": 1, "severity": "Slight"},
        {"pred": "parkinson", "conf_min": 0.50, "updrs": 2, "severity": "Mild"},
        {"pred": "parkinson", "conf_min": 0.70, "updrs": 3, "severity": "Moderate"},
        {"pred": "parkinson", "conf_min": 0.85, "updrs": 4, "severity": "Severe"},
    ],
    "metrics": {
        "test_accuracy": round(float(test_acc), 4),
        "test_f1": round(float(test_f1), 4),
        "best_val_f1": round(float(best_val_f1), 4),
    }
}

config_path = os.path.join(EXPORT_DIR, "motor_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config JSON: {config_path}")

# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📦 EXPORTED FILES:")
print(f"{'='*60}")
for f in os.listdir(EXPORT_DIR):
    fpath = os.path.join(EXPORT_DIR, f)
    fsize = os.path.getsize(fpath) / (1024 * 1024)
    print(f"   {f:45s} {fsize:8.2f} MB")
print(f"\n   Copy these to your backend:")
print(f"   motor_spiral_model.pt  → neuroverse-backend/app/ml/models/")
print(f"   motor_config.json      → neuroverse-backend/app/ml/config/")

## 1️⃣1️⃣ Backend Integration Code

Ready-to-copy Python code for `neuroverse-backend/app/ml/predictors/motor_predictor.py`.
This handles:
1. Loading the trained model
2. Predicting PD risk from spiral/wave drawings
3. Mapping to UPDRS tremor scores
4. Integrating with `fusion_service.py` motor assessment

In [ ]:
# ============================================================
# CELL 14: Backend Integration Code (copy to your backend)
# ============================================================

backend_code = '''
# ═══════════════════════════════════════════════════════════
# FILE: neuroverse-backend/app/ml/predictors/motor_predictor.py
# ═══════════════════════════════════════════════════════════
"""
Motor Drawing Predictor for NeuroVerse Backend.
Predicts PD risk from spiral/wave drawings using trained MotorNet.
Maps to UPDRS tremor scores for fusion_service.py integration.
"""
import os
import json
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from torchvision import transforms


class MotorNet(nn.Module):
    """Motor drawing classifier — must match training architecture."""
    
    def __init__(self, num_classes=2, dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, num_classes=0, global_pool='avg'
        )
        feature_dim = self.backbone.num_features  # 1280
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 256), nn.ReLU(inplace=True), nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.6),
            nn.Linear(256, 64), nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Linear(64, num_classes),
        )
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 16), nn.ReLU(inplace=True),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)
        return logits, risk.squeeze(-1)


class MotorPredictor:
    """Motor drawing predictor for NeuroVerse backend."""
    
    # Confidence → UPDRS tremor score mapping
    UPDRS_MAP = [
        # (predicted_class, min_confidence, updrs_score, label)
        ('healthy', 0.90, 0, 'Normal'),
        ('healthy', 0.70, 1, 'Slight tremor possible'),
        ('parkinson', 0.50, 2, 'Mild tremor'),
        ('parkinson', 0.70, 3, 'Moderate tremor'),
        ('parkinson', 0.85, 4, 'Severe tremor'),
    ]
    
    def __init__(self, model_path: str = None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        if model_path is None:
            model_path = os.path.join(
                os.path.dirname(__file__), '..', 'models', 'motor_spiral_model.pt'
            )
        
        checkpoint = torch.load(model_path, map_location=self.device)
        config = checkpoint['model_config']
        
        self.model = MotorNet(
            num_classes=config['num_classes'],
            dropout=config['dropout']
        ).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.img_size = config['img_size']
        norm = checkpoint['normalization']
        self.transform = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm['mean'], std=norm['std']),
        ])
        
        self.class_names = checkpoint['class_mapping']['class_names']
        self.clinical = checkpoint['clinical_thresholds']
        print(f"✅ Motor model loaded ({self.device})")
    
    def predict(self, image_path: str, drawing_type: str = 'spiral') -> dict:
        """
        Predict PD risk from a spiral/wave drawing.
        
        Returns dict with:
            - predicted_class: 'healthy' or 'parkinson'
            - confidence: float [0, 1]
            - pd_risk: float [0, 1]
            - updrs_tremor: int 0-4 (UPDRS 3.15-3.18)
            - severity_label: str
            - spiral_tremor: float [0, 1] (for fusion_service.py)
        """
        img = Image.open(image_path).convert('RGB')
        tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            logits, risk = self.model(tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred_idx = int(logits.argmax(dim=1).item())
            confidence = float(probs[pred_idx])
        
        pred_class = 'healthy' if pred_idx == 0 else 'parkinson'
        
        # Map to UPDRS tremor score
        updrs_tremor = 0
        severity_label = 'Normal'
        
        if pred_class == 'parkinson':
            if confidence >= 0.85:
                updrs_tremor, severity_label = 4, 'Severe tremor'
            elif confidence >= 0.70:
                updrs_tremor, severity_label = 3, 'Moderate tremor'
            else:
                updrs_tremor, severity_label = 2, 'Mild tremor'
        else:
            if confidence >= 0.90:
                updrs_tremor, severity_label = 0, 'Normal'
            else:
                updrs_tremor, severity_label = 1, 'Slight tremor possible'
        
        # Compute spiral_tremor (0-1 scale) for fusion_service.py
        # Maps to existing: spiral_tremor feature used in _assess_motor()
        spiral_tremor = probs[1]  # PD probability = tremor severity proxy
        
        pd_risk_info = self.clinical['pd_risk_by_prediction'].get(pred_class, {})
        
        return {
            'predicted_class': pred_class,
            'confidence': confidence,
            'class_probabilities': {str(i): float(p) for i, p in enumerate(probs)},
            'pd_risk': pd_risk_info.get('pd_risk', 0.0) * (confidence if pred_class == 'parkinson' else 1.0),
            'ad_risk': pd_risk_info.get('ad_risk', 0.0),
            'updrs_tremor': updrs_tremor,
            'severity_label': severity_label,
            'drawing_type': drawing_type,
            # Features for fusion_service.py _assess_motor():
            'spiral_tremor': float(spiral_tremor),
            'tremor_frequency': 5.0 if pred_class == 'parkinson' and confidence > 0.70 else 0.0,
            'is_pd': pred_class == 'parkinson',
        }
    
    def predict_from_bytes(self, image_bytes: bytes, drawing_type: str = 'spiral') -> dict:
        """Predict from raw image bytes (for API endpoint)."""
        import io, tempfile
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
            img.save(f.name)
            result = self.predict(f.name, drawing_type)
            os.unlink(f.name)
        return result


# ═══════════════════════════════════════════════════════════
# Usage in FastAPI:
# ═══════════════════════════════════════════════════════════
#
# predictor = MotorPredictor()
#
# @router.post("/predict/spiral-drawing")
# async def predict_spiral(file: UploadFile, drawing_type: str = "spiral"):
#     image_bytes = await file.read()
#     result = predictor.predict_from_bytes(image_bytes, drawing_type)
#     return result
#
# # The result feeds directly into fusion_service.py:
# # features = {
# #     "spiral_tremor": result["spiral_tremor"],      # → UPDRS 3.15-3.18
# #     "tremor_frequency": result["tremor_frequency"], # → PD 4-6 Hz check
# #     "spiral_duration": 10.0,                        # from app timer
# # }
'''

print(backend_code)
print("\n" + "=" * 60)
print("📋 Copy the code above to:")
print("   neuroverse-backend/app/ml/predictors/motor_predictor.py")
print("=" * 60)

## 1️⃣2️⃣ Inference Demo — Predict & Visualize

In [ ]:
# ============================================================
# CELL 15: Inference Demo — Predict & Visualize
# ============================================================

def predict_and_visualize(image_path, drawing_type='spiral', show_gradcam=True):
    """Full inference pipeline with GradCAM."""
    
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()
        risk_val = risk.item()
    
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx]
    
    # UPDRS mapping
    if pred_idx == 1:  # PD
        if confidence >= 0.85:
            updrs, severity = 4, 'Severe'
        elif confidence >= 0.70:
            updrs, severity = 3, 'Moderate'
        else:
            updrs, severity = 2, 'Mild'
    else:  # Healthy
        if confidence >= 0.90:
            updrs, severity = 0, 'Normal'
        else:
            updrs, severity = 1, 'Slight'
    
    if show_gradcam:
        targets = [ClassifierOutputTarget(pred_idx)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
        img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
        img_np = np.array(img_resized).astype(np.float32) / 255.0
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        
        # Original
        axes[0].imshow(img_pil)
        axes[0].set_title(f'Original ({drawing_type})', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # GradCAM
        axes[1].imshow(cam_image)
        axes[1].set_title('GradCAM — Tremor Regions', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        # Probability bars
        colors = ['#4ECDC4', '#FF6B6B']
        bars = axes[2].barh(
            list(CLASS_NAMES.values()),
            [probs[i] for i in range(NUM_CLASSES)],
            color=colors
        )
        axes[2].set_xlim(0, 1)
        axes[2].set_title('Class Probabilities', fontsize=12, fontweight='bold')
        for bar, p in zip(bars, probs):
            if p > 0.05:
                axes[2].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                           f'{p:.1%}', va='center', fontsize=11)
        
        title_color = 'green' if pred_idx == 0 else 'red'
        plt.suptitle(
            f"Prediction: {pred_class} (UPDRS {updrs} — {severity}) | PD Risk: {risk_val:.2f}",
            fontsize=14, fontweight='bold', color=title_color
        )
        plt.tight_layout()
        plt.show()
    
    return {
        'predicted_class': pred_class,
        'confidence': float(confidence),
        'pd_risk': float(risk_val),
        'updrs_tremor': updrs,
        'severity': severity,
        'drawing_type': drawing_type,
        'probabilities': {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    }


# ═══════════════════════════════════════════════════════════
# Test on samples from each category
# ═══════════════════════════════════════════════════════════
print("🔍 Running inference tests...\n")

for drawing_type in ['spiral', 'wave']:
    for class_name in ['healthy', 'parkinson']:
        subset = df_test_orig[
            (df_test_orig['drawing_type'] == drawing_type) &
            (df_test_orig['class_name'] == class_name)
        ]
        if len(subset) == 0:
            continue
        
        sample = subset.sample(1, random_state=SEED).iloc[0]
        print(f"{'='*60}")
        print(f"🔍 {drawing_type.upper()} — True: {class_name.upper()}")
        print(f"{'='*60}")
        result = predict_and_visualize(sample['path'], drawing_type)
        print(f"   Result: {result['predicted_class']} | UPDRS: {result['updrs_tremor']} | "
              f"Conf: {result['confidence']:.1%} | Risk: {result['pd_risk']:.2f}")
        print()

## 1️⃣3️⃣ K-Fold Cross-Validation (Robustness Check)

Since the dataset is small (~102 images), single train/test split results can be noisy.
5-fold cross-validation gives a more reliable estimate of true model performance.

In [ ]:
# ============================================================
# CELL 16: 5-Fold Cross-Validation
# ============================================================

from sklearn.model_selection import StratifiedKFold

RUN_KFOLD = True  # Set False to skip (takes ~5x training time)
K_FOLDS = 5
KFOLD_EPOCHS = 30  # Fewer epochs per fold

if RUN_KFOLD:
    print(f"{'='*60}")
    print(f"🔄 {K_FOLDS}-Fold Cross-Validation")
    print(f"{'='*60}")
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['label'])):
        print(f"\n── Fold {fold+1}/{K_FOLDS} ──")
        
        df_fold_train = df_all.iloc[train_idx]
        df_fold_val = df_all.iloc[val_idx]
        
        # Create datasets with oversampling
        fold_train_ds = ParkinsonsDrawingDataset(df_fold_train, transform=train_transforms, oversample_factor=OVERSAMPLE)
        fold_val_ds = ParkinsonsDrawingDataset(df_fold_val, transform=val_transforms)
        
        # Sampler
        fold_counts = df_fold_train['label'].value_counts().sort_index()
        fold_weights = 1.0 / fold_counts.values
        fold_sample_weights = np.tile(
            df_fold_train['label'].map(dict(zip(fold_counts.index, fold_weights))).values,
            OVERSAMPLE
        )
        fold_sampler = WeightedRandomSampler(fold_sample_weights, len(fold_sample_weights), replacement=True)
        
        fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, sampler=fold_sampler, num_workers=2, pin_memory=True, drop_last=True)
        fold_val_loader = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        # Fresh model
        fold_model = MotorNet(num_classes=2, pretrained=True, dropout=0.5).to(device)
        
        fold_bb_params = [p for n, p in fold_model.backbone.named_parameters() if p.requires_grad]
        fold_head_params = list(fold_model.classifier.parameters()) + list(fold_model.risk_head.parameters())
        fold_optimizer = optim.AdamW([
            {'params': fold_bb_params, 'lr': LEARNING_RATE / 10},
            {'params': fold_head_params, 'lr': LEARNING_RATE},
        ], weight_decay=WEIGHT_DECAY)
        fold_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(fold_optimizer, T_0=10, T_mult=2, eta_min=1e-7)
        fold_scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
        
        fold_best_f1 = 0
        fold_patience = 0
        
        for ep in range(KFOLD_EPOCHS):
            train_one_epoch(fold_model, fold_train_loader, criterion, fold_optimizer, fold_scaler, device)
            _, val_acc, val_f1, _, _, _ = validate(fold_model, fold_val_loader, criterion, device)
            fold_scheduler.step()
            
            if val_f1 > fold_best_f1:
                fold_best_f1 = val_f1
                fold_best_acc = val_acc
                fold_patience = 0
            else:
                fold_patience += 1
            
            if fold_patience >= 10:
                break
        
        # Final evaluation
        _, fold_acc, fold_f1, fold_preds, fold_labels, fold_probs = validate(
            fold_model, fold_val_loader, criterion, device
        )
        
        try:
            fold_auc = roc_auc_score(fold_labels, fold_probs[:, 1])
        except:
            fold_auc = 0.0
        
        fold_results.append({
            'fold': fold + 1,
            'accuracy': fold_best_acc,
            'f1': fold_best_f1,
            'auc': fold_auc,
        })
        
        print(f"   Fold {fold+1}: Acc={fold_best_acc:.3f} | F1={fold_best_f1:.3f} | AUC={fold_auc:.3f}")
        
        del fold_model, fold_optimizer
        torch.cuda.empty_cache() if device.type == 'cuda' else None
    
    # Summary
    df_folds = pd.DataFrame(fold_results)
    print(f"\n{'='*60}")
    print(f"📊 {K_FOLDS}-FOLD CROSS-VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"   Accuracy: {df_folds['accuracy'].mean():.3f} ± {df_folds['accuracy'].std():.3f}")
    print(f"   F1 Score: {df_folds['f1'].mean():.3f} ± {df_folds['f1'].std():.3f}")
    print(f"   ROC-AUC:  {df_folds['auc'].mean():.3f} ± {df_folds['auc'].std():.3f}")
    print(f"\n   Per-fold:")
    for _, r in df_folds.iterrows():
        print(f"     Fold {int(r['fold'])}: Acc={r['accuracy']:.3f} F1={r['f1']:.3f} AUC={r['auc']:.3f}")
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(K_FOLDS)
    width = 0.25
    ax.bar(x - width, df_folds['accuracy'], width, label='Accuracy', color='#4ECDC4')
    ax.bar(x, df_folds['f1'], width, label='F1', color='#FF6B6B')
    ax.bar(x + width, df_folds['auc'], width, label='AUC', color='#45B7D1')
    ax.set_xlabel('Fold')
    ax.set_ylabel('Score')
    ax.set_title(f'{K_FOLDS}-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i+1}' for i in range(K_FOLDS)])
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'motor_kfold_results.png'), dpi=150, bbox_inches='tight')
    plt.show()

else:
    print("⏭️  Skipping K-Fold cross-validation")

---

## ✅ Summary — Motor Spiral Drawing Model Complete!

### What was trained:
| Component | Details |
|-----------|---------|
| **Model** | MotorNet (EfficientNet-B0 + custom PD head) |
| **Datasets** | 5 combined sources (~800+ unique images) |
| **Sources** | Kaggle: kmader, alihussain, vasantha + GitHub: HandPD, NewHandPD |
| **Task** | Binary: Healthy vs Parkinson's Disease |
| **Drawing Types** | Spiral + Wave + Meander (all motor fluidity tasks) |
| **Augmentation** | 12 transforms with adaptive oversampling (3-10x based on dataset size) |
| **Deduplication** | MD5 content hash to remove identical images across datasets |
| **Training** | AdamW + Cosine Annealing + Label Smoothing (0.15) |
| **XAI** | GradCAM heatmaps showing tremor regions |
| **Validation** | 5-Fold cross-validation for robustness |

### Multi-dataset sources:
| Dataset | Source | Images | Paper |
|---------|--------|--------|-------|
| Parkinson's Drawings | Kaggle (kmader) | ~102 | Zham et al., 2017 |
| Parkinson Drawing Spiral+Wave | Kaggle (alihussain) | ~200+ | Community |
| Spiral Drawings PD | Kaggle (vasantha) | ~150+ | Community |
| HandPD | GitHub (biolab) | ~92 | Pereira et al., 2016 |
| NewHandPD | GitHub (biolab) | ~264 | Pereira et al., 2016 |

### Exported files:
| File | Destination | Purpose |
|------|------------|---------|
| `motor_spiral_model.pt` | `app/ml/models/` | Backend inference |
| `motor_spiral_model_mobile.pt` | — | Mobile/edge deployment |
| `motor_config.json` | `app/ml/config/` | UPDRS mapping + thresholds |

### How it fits in NeuroVerse:
```
Flutter App: User draws spiral/wave on Canvas
    ↓ PNG image + stroke data (speed, pressure, tremor)
FastAPI Backend: MotorPredictor.predict(image)
    ↓ {predicted_class: "parkinson", pd_risk: 0.72, updrs_tremor: 3}
fusion_service.py _assess_motor():
    ↓ Combines spiral_tremor + tapping_rate + tremor_frequency
    ↓ Calculates Hoehn & Yahr stage
    ↓ UPDRS motor total score
xai_service.py: GradCAM tremor heatmap
    ↓ Visual explanation for doctor dashboard
```

### Integration with fusion_service.py:
The model output maps directly to existing features:
- `spiral_tremor` → `features["spiral_tremor"]` (UPDRS 3.15-3.18)
- `tremor_frequency` → `features["tremor_frequency"]` (4-6 Hz = PD range)
- `pd_risk` → used in `_assess_motor()` return dict

### NeuroVerse PD Detection Triad:
- [x] **Speech** — DementiaBank + EWA-DB ✅
- [x] **Clock Drawing** — Roboflow CDT ✅
- [x] **Motor/Spiral** — Multi-dataset (5 sources, ~800+ images) ✅ (THIS NOTEBOOK)
- [ ] **Gait** — Walking + Balance + Turn-in-Place
- [ ] **Fusion** — Combine all module scores